In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [17]:
# --------------------------------
# 1) Load the cleaned dataset
# --------------------------------
CSV_PATH = r'C:\Users\SAKARIYE JAMA\Desktop\new2\clean_car_dataset.csv'
df = pd.read_csv(CSV_PATH)

df.head()



,Price,Odometer_km,Doors,Accidents,Year,Location_City,Location_Rural,Location_Suburb,CarAge,Is_Urban,Km_per_year,LogPrice
0,1500.0,0.128390,0.254091,0.316968,-1.686714,1,0,0,1.686714,1,-0.615631,7.313887
1,4171.0,-0.044709,0.254091,-0.820867,0.794617,0,1,0,-0.794617,0,0.070446,8.336151
2,5331.0,-0.440923,0.254091,-0.820867,0.518913,0,0,1,-0.518913,1,-0.267993,8.581482
3,1500.0,0.203135,0.254091,0.316968,-1.548862,0,0,1,1.548862,1,-0.587024,7.313887
4,1500.0,-0.044709,-0.931668,-0.820867,1.621727,1,0,0,-1.621727,1,1.738196,7.313887


In [5]:
# --------------------------------
# 2) Split features (X) and target (y)
# --------------------------------
# We predict "Price". We also drop "LogPrice" from X so we don't leak target info.
X = df.drop(columns=["Price", "LogPrice"])
y = df["Price"]

In [6]:

# 3) Train/test split for fair evaluation
#####################################################################################
# Keep 20% of data for testing generalization. random_state for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
# 4) Train Linear Regression
# --------------------------------
# Linear model is simple and interpretable; good baseline.
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

In [8]:
# 5) Train Random Forest
# --------------------------------
# Ensemble model captures non-linear relationships; often stronger than linear.
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [9]:
# 6) Helper to print metrics nicely
# --------------------------------
def print_metrics(name, y_true, y_pred):
    """Print R², MAE, MSE, RMSE for a model's predictions."""
    r2  = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f"\n{name} Performance:")
    print(f"  R²   : {r2:.3f}")          # higher is better (max = 1.0)
    print(f"  MAE  : {mae:,.0f}")        # lower is better (absolute error)
    print(f"  MSE  : {mse:,.0f}")        # lower is better (squared error)
    print(f"  RMSE : {rmse:,.0f}")       # lower is better (same units as Price)

In [10]:
# 7) Show results for both models
# --------------------------------
print_metrics("Linear Regression", y_test, lr_pred)
print_metrics("Random Forest",   y_test, rf_pred)


Linear Regression Performance:
  R²   : 0.436
  MAE  : 1,428
  MSE  : 3,755,299
  RMSE : 1,938

Random Forest Performance:
  R²   : 0.265
  MAE  : 1,208
  MSE  : 4,889,052
  RMSE : 2,211


### Observation: Why Random Forest underperforms Linear Regression here

Random Forest performed *worse* than Linear Regression on this dataset (R² = 0.265 vs 0.436), which is the opposite of what we'd usually expect, since Random Forest is normally the more powerful, flexible model.

The likely reason is **dataset size**. This dataset only has 140 rows total, so after the 80/20 split, the model trains on just 112 rows and is tested on 28. Random Forest usually needs a larger amount of data to learn reliable patterns across its many decision trees; with so few training rows, it struggles to generalize and can end up fitting noise instead of the true relationship. Linear Regression, on the other hand, tends to perform reasonably well on small datasets when the underlying relationship between the features and the target is close to linear, which is likely the case here.

In [16]:
# 8) Single-row prediction (sanity check)
# --------------------------------
# Pick one unseen row from X_test and predict both models.
# Use iloc[[i]] (double brackets) to keep it as a DataFrame with column names
i = 3

x_one_df = X_test.iloc[[i]]   # 1-row DataFrame (keeps feature names)
y_true   = y_test.iloc[i]     # scalar
p_lr_one = float(lr.predict(x_one_df)[0])
p_rf_one = float(rf.predict(x_one_df)[0])

print("\nSingle-row sanity check:")
print(f"  Actual Price: ${y_true:,.0f}")
print(f"  LR Pred     : ${p_lr_one:,.0f}")
print(f"  RF Pred     : ${p_rf_one:,.0f}")


Single-row sanity check:
  Actual Price: $7,009
  LR Pred     : $5,703
  RF Pred     : $6,941
